# Car listings: EDA and temporal validation
This notebook consumes pipeline outputs. The bundled data is synthetic and must not be interpreted as current market evidence.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data = pd.read_csv(ROOT / 'data/processed/listings.csv', parse_dates=['listing_date'])
data.info()

In [ ]:
display(data.describe(include='all').T)
display(data.isna().mean().sort_values(ascending=False).to_frame('missing_rate'))
display(data.groupby(['make', 'model']).agg(rows=('asking_price','size'), median_price=('asking_price','median')).sort_values('rows', ascending=False))

In [ ]:
monthly = data.set_index('listing_date').groupby('model')['asking_price'].resample('ME').median().unstack(0)
monthly.plot(figsize=(12, 5), title='Monthly median asking price by model')
plt.ylabel('million IRR'); plt.show()

In [ ]:
metrics = json.loads((ROOT / 'artifacts/metrics.json').read_text())
pd.DataFrame({'model': metrics['model'], 'baseline': metrics['baseline_make_model_median']})

## Interpretation
Check segment counts before trusting aggregate metrics. Temporal error and interval coverage matter more than random-split scores. The direct label is asking price; the transaction adjustment remains an unverified scenario.